<a href="https://colab.research.google.com/github/Haross/sql_midterms/blob/main/final_term_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **BTM-3300 Finalterm Practice**

**Take your time to **read each prompt carefully**, translate it into logic, and then into SQL.**


## **SQL Environment Setup (do not edit)**

In [1]:
# @title

%%capture
!mkdir -p notebook_lib
!wget --cache=off -q -O notebook_lib/sql_runner.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner.py
!wget --cache=off -q -O notebook_lib/sql_runner_store.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_store.py
!wget --cache=off -q -O notebook_lib/sql_runner_ui_bits.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/sql_runner_ui_bits.py
!wget --cache=off -q -O notebook_lib/validators.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/validators.py

!wget --cache=off -q -O notebook_lib/cloud_submitter.py \
  https://raw.githubusercontent.com/Haross/sql_notebook/main/notebook_lib/cloud_submitter.py

!touch notebook_lib/__init__.py
#!pip install -q duckdb

from notebook_lib.sql_runner import make_sql_runner
from notebook_lib.validators import make_df_validator_nospoilers, check_process_rules
from notebook_lib.cloud_submitter import make_cloud_run_submitter
import duckdb
import pandas as pd
from pathlib import Path


In [2]:
# @title

DB_FILE = 'class_joins.duckdb'

if DB_FILE != ":memory:":
    Path(DB_FILE).unlink(missing_ok=True)

conn = duckdb.connect(DB_FILE)

conn.execute(r'''
-- Drop tables if they already exist
DROP TABLE IF EXISTS students;

-- Create tables
CREATE TABLE students (
    student_age INTEGER,
    grade TEXT,
    attendance INTEGER,
    extra_credit BOOLEAN,
    program TEXT,
    advisor_note TEXT,
    disciplinary_flag BOOLEAN
);

INSERT INTO students VALUES

(19, 'A', 90, FALSE, 'science', 'excellent', FALSE),
(23, 'C', 85, TRUE,  'math',    'good',      FALSE),
(25, 'B', 96, FALSE, 'arts',    'recommended student', FALSE),

(20, 'C', 82, FALSE, 'science', 'ok', FALSE),                -- C without extra_credit
(26, 'A', 70, FALSE, 'math',    'good', FALSE),              -- attendance too low
(22, 'B', 88, FALSE, 'arts',    'average', FALSE),           -- arts but not >90 or recommended

(21, 'B', 85, FALSE, 'science', 'good', TRUE),               -- disciplinary, not grade A
(24, 'B', 97, FALSE, 'arts',    'recommended', TRUE),        -- arts but no "approved"
(23, 'A', 88, FALSE, 'math',    'good', TRUE),               -- math has no exception

(20, 'A', 85, FALSE, 'science', 'good', TRUE),               -- science + A
(25, 'B', 96, FALSE, 'arts',    'approved case', TRUE),      -- arts + approved + >95

(23, 'B', 92, FALSE, 'arts',    'probation - low', FALSE),   -- probation, no extra_credit
(24, 'C', 85, TRUE,  'science', 'probation warning', FALSE), -- extra_credit but not math or B>22
(23, 'A', 88, TRUE,  'math',    'probation case', FALSE),    -- math + extra_credit
(24, 'B', 90, TRUE,  'science', 'probation note', FALSE),    -- age>22 + B + extra_credit
(25, 'B', 90, FALSE, 'science', 'recommended', TRUE),
(23, 'A', 95, TRUE,  'math', 'probation issue', FALSE),
(26, 'C', 100, TRUE, 'math', 'recommended and approved', FALSE);

DROP TABLE IF EXISTS shipments;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS discounts;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS employees;
DROP TABLE IF EXISTS departments;

CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name VARCHAR
);

CREATE TABLE employees (
    employee_id INTEGER PRIMARY KEY,
    name VARCHAR,
    department_id INTEGER,
    manager_id INTEGER,
    salary DECIMAL(10,2),
    hire_date DATE,
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (manager_id) REFERENCES employees(employee_id)
);

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name VARCHAR,
    city VARCHAR
);

CREATE TABLE discounts (
    discount_id INTEGER PRIMARY KEY,
    discount_percent DECIMAL(5,2),
    description VARCHAR
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name VARCHAR,
    category VARCHAR,
    price DECIMAL(10,2)
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id INTEGER,
    discount_id INTEGER,
    order_amount DECIMAL(10,2),
    order_date DATE,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id),
    FOREIGN KEY (discount_id) REFERENCES discounts(discount_id)
);

CREATE TABLE shipments (
    shipment_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    shipment_date DATE,
    carrier VARCHAR,
    FOREIGN KEY (order_id) REFERENCES orders(order_id)
);

INSERT INTO departments (department_id, department_name) VALUES
(1, 'HR'),
(2, 'Engineering'),
(3, 'Sales'),
(4, 'Finance'),
(5, 'Marketing');

-- managers first
INSERT INTO employees (employee_id, name, department_id, manager_id, salary, hire_date) VALUES
(1,  'Alice',  2, NULL, 120000.00, DATE '2017-01-10'),
(7,  'Grace',  3, NULL, 110000.00, DATE '2016-09-01'),
(12, 'Leo',    4, NULL, 105000.00, DATE '2015-04-22'),
(15, 'Oscar',  1, NULL,  98000.00, DATE '2017-06-30'),
(18, 'Rita',   5, NULL, 100000.00, DATE '2018-02-01');

-- employees who report to top-level managers
INSERT INTO employees (employee_id, name, department_id, manager_id, salary, hire_date) VALUES
(2,  'Bob',    2, 1,     90000.00, DATE '2018-03-15'),
(3,  'Carol',  2, 1,    130000.00, DATE '2019-06-01'),
(6,  'Frank',  2, 1,     88000.00, DATE '2022-11-03'),
(8,  'Henry',  3, 7,     70000.00, DATE '2019-01-10'),
(9,  'Irene',  3, 7,    115000.00, DATE '2020-05-05'),
(11, 'Karen',  3, 7,     73000.00, DATE '2022-12-01'),
(13, 'Mona',   4, 12,    95000.00, DATE '2018-10-10'),
(14, 'Nina',   4, 12,    97000.00, DATE '2021-03-17'),
(16, 'Paula',  1, 15,    65000.00, DATE '2019-09-14'),
(17, 'Quinn',  1, 15,   102000.00, DATE '2020-01-08'),
(19, 'Sam',    5, 18,    72000.00, DATE '2020-06-11'),
(20, 'Tina',   5, 18,    76000.00, DATE '2021-11-25'),
(21, 'Uma',    3, 1,    125000.00, DATE '2023-01-15'),
(22, 'Victor', 5, 7,    112000.00, DATE '2023-04-20');

-- employees who report to non-top-level managers
INSERT INTO employees (employee_id, name, department_id, manager_id, salary, hire_date) VALUES
(4,  'David',  2, 2,     85000.00, DATE '2020-02-20'),
(5,  'Eva',    2, 2,     91000.00, DATE '2021-07-12'),
(10, 'Jack',   3, 8,     68000.00, DATE '2021-08-18');

INSERT INTO customers (customer_id, customer_name, city) VALUES
(1, 'Acme Corp', 'Madrid'),
(2, 'Blue Retail', 'Barcelona'),
(3, 'City Market', 'Valencia'),
(4, 'Delta Stores', 'Seville'),
(5, 'Evergreen Ltd', 'Bilbao'),
(6, 'Fresh Foods', 'Malaga'),
(7, 'Global Tech', 'Lisbon'),
(8, 'Home Style', 'Porto');

INSERT INTO discounts (discount_id, discount_percent, description) VALUES
(1,  5.00, 'Seasonal 5%'),
(2, 10.00, 'Loyalty 10%'),
(3, 15.00, 'Promotion 15%');

INSERT INTO products (product_id, product_name, category, price) VALUES
(1, 'Laptop',      'Electronics', 1200.00),
(2, 'Mouse',       'Electronics',   25.00),
(3, 'Keyboard',    'Electronics',   45.00),
(4, 'Desk Chair',  'Furniture',    180.00),
(5, 'Notebook',    'Office',         5.00),
(6, 'Printer',     'Electronics',  250.00),
(7, 'Desk',        'Furniture',    320.00),
(8, 'Monitor',     'Electronics',  300.00);

INSERT INTO orders (order_id, customer_id, product_id, discount_id, order_amount, order_date) VALUES
(1,  1, 1, 2, 1080.00, DATE '2025-01-10'),
(2,  1, 2, NULL,  25.00, DATE '2025-01-11'),
(3,  1, 3, NULL,  45.00, DATE '2025-01-12'),
(4,  1, 8, 1,   285.00, DATE '2025-01-13'),
(5,  2, 1, NULL, 1200.00, DATE '2025-01-14'),
(6,  2, 4, 3,    153.00, DATE '2025-01-15'),
(7,  2, 5, NULL,   5.00, DATE '2025-01-16'),
(8,  3, 6, 2,    225.00, DATE '2025-01-17'),
(9,  3, 2, NULL,  25.00, DATE '2025-01-18'),
(10, 4, 7, NULL, 320.00, DATE '2025-01-19'),
(11, 4, 4, NULL, 180.00, DATE '2025-01-20'),
(12, 4, 5, 1,      4.75, DATE '2025-01-21'),
(13, 4, 3, NULL,  45.00, DATE '2025-01-22'),
(14, 5, 1, 3,   1020.00, DATE '2025-01-23'),
(15, 5, 8, NULL, 300.00, DATE '2025-01-24'),
(16, 6, 2, NULL,  25.00, DATE '2025-01-25'),
(17, 6, 2, NULL,  25.00, DATE '2025-01-26'),
(18, 6, 3, NULL,  45.00, DATE '2025-01-27'),
(19, 7, 1, NULL, 1200.00, DATE '2025-01-28'),
(20, 7, 6, 1,    237.50, DATE '2025-01-29'),
(21, 7, 8, NULL, 300.00, DATE '2025-01-30'),
(22, 7, 2, NULL,  25.00, DATE '2025-01-31'),
(23, 8, 5, NULL,   5.00, DATE '2025-02-01'),
(24, 8, 5, NULL,   5.00, DATE '2025-02-02'),
(25, 8, 5, NULL,   5.00, DATE '2025-02-03'),
(26, 8, 4, 2,    162.00, DATE '2025-02-04'),
(27, 1, 2, NULL,  25.00, DATE '2025-02-05'),
(28, 2, 2, NULL,  25.00, DATE '2025-02-06'),
(29, 3, 2, NULL,  25.00, DATE '2025-02-07'),
(30, 4, 2, NULL,  25.00, DATE '2025-02-08'),
(31, 5, 2, NULL,  25.00, DATE '2025-02-09'),
(32, 6, 2, NULL,  25.00, DATE '2025-02-10'),
(33, 7, 2, NULL,  25.00, DATE '2025-02-11'),
(34, 8, 2, NULL,  25.00, DATE '2025-02-12');

INSERT INTO shipments (shipment_id, order_id, shipment_date, carrier) VALUES
(1,  1,  DATE '2025-01-12', 'DHL'),
(2,  2,  DATE '2025-01-13', 'UPS'),
(3,  3,  DATE '2025-01-14', 'DHL'),
(4,  5,  DATE '2025-01-16', 'FedEx'),
(5,  6,  DATE '2025-01-17', 'DHL'),
(6,  8,  DATE '2025-01-19', 'UPS'),
(7,  10, DATE '2025-01-21', 'DHL'),
(8,  11, DATE '2025-01-22', 'DHL'),
(9,  12, DATE '2025-01-23', 'FedEx'),
(10, 14, DATE '2025-01-25', 'UPS'),
(11, 19, DATE '2025-01-30', 'DHL'),
(12, 20, DATE '2025-01-31', 'UPS'),
(13, 22, DATE '2025-02-02', 'FedEx'),
(14, 23, DATE '2025-02-03', 'DHL'),
(15, 26, DATE '2025-02-05', 'UPS'),
(16, 27, DATE '2025-02-06', 'DHL'),
(17, 28, DATE '2025-02-07', 'DHL'),
(18, 30, DATE '2025-02-09', 'FedEx'),
(19, 32, DATE '2025-02-11', 'UPS'),
(20, 34, DATE '2025-02-13', 'DHL');
''')
print(f"Duckdb database ready ✅ ({DB_FILE})")


Duckdb database ready ✅ (class_joins.duckdb)


In [3]:
# @title Cloud submission config (do not edit)
CLOUD_SUBMIT_URL = 'https://cinemax-validator-830800716261.europe-west1.run.app/submit'
CLOUD_EXAM_ID = 'FINAL_TERM_PRACTICE_2026'
CLOUD_API_KEY = 'SECRET123'


# Token

In [4]:
# @title
import ipywidgets as widgets
from IPython.display import display, HTML
from pathlib import Path

TOKEN_FILE = Path("student_token.txt")

status_out = widgets.Output()

def update_status(title, message, type="info"):
    icons = {
        "info": "ℹ️",
        "success": "✅",
        "warning": "⚠️",
        "error": "❌",
    }

    icon = icons.get(type, "ℹ️")

    with status_out:
        status_out.clear_output()
        display(HTML(f"""
        <style>
        /* ===== Theme Variables ===== */
        html {{
            --info-border: #1a73e8;
            --info-bg: #eef3fe;

            --success-border: #2e7d32;
            --success-bg: #eaf7ee;

            --warning-border: #f9a825;
            --warning-bg: #fff8e1;

            --error-border: #c62828;
            --error-bg: #fdecea;

            --text-color: #202124;
            --shadow: 0 2px 6px rgba(0,0,0,0.05);
        }}

        html[theme="dark"] {{
            --info-border: #8ab4f8;
            --info-bg: #1e2a3a;

            --success-border: #81c995;
            --success-bg: #1f3324;

            --warning-border: #fdd663;
            --warning-bg: #3a3218;

            --error-border: #f28b82;
            --error-bg: #3a1f1f;

            --text-color: #e8eaed;
            --shadow: 0 2px 8px rgba(0,0,0,0.6);
        }}

        /* ===== Status Box ===== */
        .status-box {{
            padding:16px;
            border-left:6px solid var(--{type}-border);
            background: var(--{type}-bg);
            border-radius:10px;
            font-family:Arial, sans-serif;
            line-height:1.6;
            box-shadow: var(--shadow);
            color: var(--text-color);
        }}

        .status-title {{
            font-size:16px;
            font-weight:600;
            margin-bottom:6px;
        }}

        .status-message {{
            font-size:14px;
        }}
        </style>

        <div class="status-box">
            <div class="status-title">
                {icon} {title}
            </div>
            <div class="status-message">
                {message}
            </div>
        </div>
        """))

display(status_out)


t = TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else ""

update_status(
    "Important: Before Starting",
    """
    The next cell will ask you to enter your <b>student token</b>.<br><br>
    After typing it, press <b>Enter</b>.<br>
    You only need to do this once.
    """,
    type="info"
)

Output()

In [5]:
# @title
from pathlib import Path

TOKEN_FILE = Path("student_token.txt")

existing = TOKEN_FILE.read_text().strip() if TOKEN_FILE.exists() else ""
if not existing:
    while True:
        token = input("Student token: ").strip()
        if token:
            break
        print("⚠️ Token cannot be empty. Try again.")

    TOKEN_FILE.write_text(token)
    update_status(
        "Token Saved Successfully",
        f"Your token has been stored securely.<br><br>Your token is:<b>{token}</b>",
        type="success"
    )
    print("✅ Token saved.")
else:
    token = existing
    update_status(
        "Token Already Saved",
        f"Your token has been already stored securely.<br><br>Your token is: <b>{token}</b>",
        type="success"
    )
    print("✅ Token already saved.")

Student token: prueba
✅ Token saved.


In [6]:
# @title ER diagram
%%html
<img id="er-img" style="width:50%; max-width:100%;  height:auto;"
     data-light="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/ER_final_term_practice.png"
     data-dark="https://raw.githubusercontent.com/Haross/DB_pics_nt/main/ER_final_term_practice_black.png"
     alt="ER diagram">

<script>
  const img = document.getElementById("er-img");

  function isDarkTheme() {
    // Colab sets html[theme=dark] on the top document
    const themeAttr = document.documentElement.getAttribute("theme");
    if (themeAttr) return themeAttr === "dark";

    // fallback: OS/browser preference
    return window.matchMedia && window.matchMedia("(prefers-color-scheme: dark)").matches;
  }

  function updateImage() {
    img.src = isDarkTheme() ? img.dataset.dark : img.dataset.light;
  }

  updateImage();

  // React to Colab theme toggles (attribute changes)
  new MutationObserver(updateImage).observe(document.documentElement, {
    attributes: true,
    attributeFilter: ["theme"]
  });

  // React to OS/browser theme changes (fallback)
  if (window.matchMedia) {
    const mq = window.matchMedia("(prefers-color-scheme: dark)");
    mq.addEventListener?.("change", updateImage);
    mq.addListener?.(updateImage); // older browsers
  }
</script>

In [7]:
# @title Question 1 (10 points)
submitter_question_1 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='q1',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="question_1",
    description_md='### Question 1\nFind departments where:\n\n* the average salary is **higher than the average salary of all departments**\n* AND the department has **at least 5 employees**\n\nReturn: department_name and avg_salary\n',
    submitter=submitter_question_1,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [8]:
# @title Question 2 (10 points)
submitter_question_2 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='q2',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="question_2",
    description_md='### Question 2\nFind employees who:\n\n* earn more than their manager\n* AND work in a **different department than their manager**          \n\nReturn: employee_name and manager_name\n',
    submitter=submitter_question_2,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [9]:
# @title Question 3 (10 points)
submitter_question_3 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='q3',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="question_3",
    description_md='### Question 3\nFind customers who:\n\n* placed **at least one order**\n* but **none of their orders have been shipped**\n\nReturn: customer_name\n',
    submitter=submitter_question_3,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [10]:
# @title Question 4 (10 points)
submitter_question_4 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='q4',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="question_4",
    description_md='### Question 4\nFind products where:\n\n* total orders > 10\n* AND less than 50% of them have been shipped\n\nReturn: product_name, total_orders, shipped_orders\n',
    submitter=submitter_question_4,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [11]:
# @title Question 5 (10 points)
submitter_question_5 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='q5',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="question_5",
    description_md='### Question 5\nFind pairs of employees who:\n\n* work in the same department\n* AND have the **same salary**\n\nReturn: employee_1 and employee_2\n',
    submitter=submitter_question_5,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [12]:
# @title Question 6 (10 points)
submitter_question_6 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='q6',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="question_6",
    description_md='### Question 6\nFind customers who:\n\n* placed more than 3 orders\n* AND have **at least one unshipped order**\n* AND whose **average order value is above the global average order value**\n\nReturn: customer_name and total_orders\n',
    submitter=submitter_question_6,
    sol_sql=None,
    select_only=True,
    dedupe=True
)


In [13]:
# @title Question 7 (25 points)
submitter_question_7 = make_cloud_run_submitter(
    submit_url=CLOUD_SUBMIT_URL,
    exam_id=CLOUD_EXAM_ID,
    question_id='q7',
    api_key=CLOUD_API_KEY,
)

make_sql_runner(
    conn,
    runner_id="question_7",
    description_md='### Question 7\n#🎓 Student Eligibility Report (25 points)\n\nWrite a query that returns all students who satisfy a set of academic and administrative conditions.\n\n### Main filtering\n\nWrite a query that selects students who meet **at least one** of the following profiles:\n\n* Students in the **science** or **math** program who:\n\n    * have attendance between **80 and 100 (inclusive)**, and\n    * either have grade **A or B**,\n    * or have grade **C** with extra_credit = TRUE\n\n\n* Students in the **arts** program who:\n\n\n    * are **older than 20**, and\n    * either have attendance greater than **90**,\n    * or have an advisor_note that contains the word **"recommended"**\n\n---\n### Disciplinary filtering \n\nReject students with disciplinary_flag = TRUE unless:\n\n* the student is in the **science** program with grade **A**,\n* or the student is in the **arts** program and:\n\n    * the advisor_note contains **"approved"**, and\n    * attendance is greater than **95**\n\n---\n### Advisor note filtering\n\nReject students whose advisor_note starts with **"probation"** unless:\n\n* extra_credit = TRUE, and:\n\n    * the student is in the **math** program,\n    * or the student is **older than 22** and has grade **B**\n',
    submitter=submitter_question_7,
    sol_sql=None,
    select_only=True,
    dedupe=True
)
